In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import kagglehub
import pandas as pd
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, T5Tokenizer, T5ForConditionalGeneration

import OpenAttack as oa
import torch.nn.functional as F
from datasets import Dataset, load_dataset
import numpy as np
from tqdm import tqdm

c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("imdb", split="test")

df = dataset.to_pandas()
df.head()

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 216336.39 examples/s]


,text,label
0,I love sci-fi and am willing to put up with a ...,0
1,"Worth the entertainment value of a rental, esp...",0
2,its a totally average film with a few semi-alr...,0
3,STAR RATING: ***** Saturday Night **** Friday ...,0
4,"First off let me say, If you haven't enjoyed a...",0


In [3]:
# Standardize
df.columns = [c.lower() for c in df.columns]

TEXT_COL = "text"
LABEL_COL = "label"

df = df.dropna(subset=[TEXT_COL, LABEL_COL])
df = df[[TEXT_COL, LABEL_COL]]


In [6]:
test_ds = dataset.select(range(100))

formatted_data = [
    {"x": test_ds["text"][i], "y": test_ds["label"][i]}
    for i in range(min(50, len(test_ds["text"])))
]

In [7]:
def load_model(path):
    # detect model type from path
    if "roberta" in path:
        tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    else:
        tokenizer = AutoTokenizer.from_pretrained(
            path,
            local_files_only=True
        )

    model = AutoModelForSequenceClassification.from_pretrained(
        path,
        low_cpu_mem_usage=True,
        local_files_only=True
    )

    return model, tokenizer

In [8]:
class TransformerClassifier(oa.Classifier):
    def __init__(self, model, tokenizer, device="cpu"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_prob(self, input_):
        inputs = self.tokenizer(
            input_,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
        return outputs.logits.softmax(dim=1).cpu().numpy()

    def get_pred(self, input_):
        return self.get_prob(input_).argmax(axis=1)

In [24]:
attackers = {
    "textFooler (SEA)": oa.attackers.TextFoolerAttacker(),
    "SCPN": oa.attackers.SCPNAttacker(),
    "GAN": oa.attackers.GANAttacker()
}

In [25]:
def run_attackeval(model, tokenizer, dataset, attacker, num_samples=50):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    clf = TransformerClassifier(model, tokenizer, device=device)
    attack_eval = oa.AttackEval(attacker,clf)
    results = attack_eval.eval(dataset, visualize=True)

In [26]:
trained_model = {}

trained_model["bert"] = (*load_model("./imdb/bert"), test_ds)
trained_model["electra"] = (*load_model("./imdb/electra"), test_ds)
trained_model["roberta"] = (*load_model("./imdb/roberta"), test_ds)

In [ ]:
all_results = {}

for name, (model, tokenizer, test_data) in trained_model.items():
    print(f"\n========== {name.upper()} ==========\n")

    for name, attacker in attackers.items():
        print(f"\n{name.upper()}")

        results = run_attackeval(
            model,
            tokenizer,
            formatted_data, 
            attacker,
            num_samples=50 
        )


    all_results[name] = results


========== BERT ==========


TEXTFOOLER (SEA)
Sample: 1 =====================================================================
Label: 0 (98.93%) --> Failed!               |                                   
                                            |                                   
I love sci - fi and am willing to put up    |                                   
with a lot . Sci - fi movies / TV are       |                                   
usually underfunded , under - appreciated   |                                   
and misunderstood . I tried to like this ,  |                                   
I really did , but it is to good TV sci -   |                                   
fi as Babylon 5 is to Star Trek ( the       |                                   
original ). Silly prosthetics , cheap       |                                   
cardboard sets , stilted dialogues , CG     |                                   
that doesn ' t match the background , and   |                  

Exception when evaluate data {'x': "STAR RATING: ***** Saturday Night **** Friday Night *** Friday Morning ** Sunday Night * Monday Morning <br /><br />Former New Orleans homicide cop Jack Robideaux (Jean Claude Van Damme) is re-assigned to Columbus, a small but violent town in Mexico to help the police there with their efforts to stop a major heroin smuggling operation into their town. The culprits turn out to be ex-military, lead by former commander Benjamin Meyers (Stephen Lord, otherwise known as Jase from East Enders) who is using a special method he learned in Afghanistan to fight off his opponents. But Jack has a more personal reason for taking him down, that draws the two men into an explosive final showdown where only one will walk away alive.<br /><br />After Until Death, Van Damme appeared to be on a high, showing he could make the best straight to video films in the action market. While that was a far more drama oriented film, with The Shepherd he has returned to the high-k

Sample: 4 =====================================================================
Label: 1 (98.20%) --> Failed!               |                                   
                                            |                                   
STAR RATING : ***** Saturday Night ****     |                                   
Friday Night *** Friday Morning ** Sunday   |                                   
Night * Monday Morning < br />< br />       |                                   
Former New Orleans homicide cop Jack        |                                   
Robideaux ( Jean Claude Van Damme ) is re - |                                   
assigned to Columbus , a small but violent  |                                   
town in Mexico to help the police there     |                                   
with their efforts to stop a major heroin   |                                   
smuggling operation into their town . The   |                                   
culprits turn out to be ex - 

Exception when evaluate data {'x': 'I saw the Mogul Video VHS of this. That\'s another one of those old 1980s distributors whose catalog I wish I had!<br /><br />This movie was pretty poor. Though retitled "Don\'t Look in the Attic," the main admonition that is repeated in this is "Don\'t go to the villa." Just getting on the grounds of the villa is a bad idea. A character doesn\'t go into the attic until an hour into the movie, and actually should have done it earlier because of what is learned there.<br /><br />The movie starts in Turin, Italy in the 1950s. Two men are fighting, and a woman is telling them the villa is making them do it. One man kills the other, then regrets it, and the woman pulls out the knife and stabs him with it. She flees the villa, and after she\'s left a chair moves by itself (what\'s the point of that?), but when in the garden a hand comes up through the ground and drags he into the earth.<br /><br />From there, it\'s the present day, thirty years later. The

Loading parser from serialized file d:\DL project\Post_midsem\data\TProcess.StanfordParser\englishPCFG.ser.gz ... done [0.7 sec].
Parsing file: C:\Users\Priyanka\AppData\Local\Temp\tmpnowm9akd
Parsing [sent. 1 len. 476]: I saw the Mogul Video VHS of this . That 's another one of those old 1980s distributors whose catalog I wish I had ! <br /> <br /> This movie was pretty poor . Though retitled `` Do n't Look in the Attic , '' the main admonition that is repeated in this is `` Do n't go to the villa . '' Just getting on the grounds of the villa is a bad idea . A character does n't go into the attic until an hour into the movie , and actually should have done it earlier because of what is learned there . <br /> <br /> The movie starts in Turin , Italy in the 1950s . Two men are fighting , and a woman is telling them the villa is making them do it . One man kills the other , then regrets it , and the woman pulls out the knife and stabs him with it . She flees the villa , and after she 's 

...

Loading parser from serialized file d:\DL project\Post_midsem\data\TProcess.StanfordParser\englishPCFG.ser.gz ... done [0.7 sec].
Parsing file: C:\Users\Priyanka\AppData\Local\Temp\tmpkqzy75mu
Parsing [sent. 1 len. 423]: This movie had a very unique effect on me : it stalled my realization that this movie REALLY sucks ! It is disguised as a `` thinker 's film '' in the likes of Memento and other jewels like that , but at the end , and even after a few minutes , you come to realize that this is nothing but utter pretentious cr4p . Probably written by some collage student with friends to compassionate to tell him that his writing sucks . The whole idea is ... I do n't even know if it tried to scratch on the supernatural , or they want us to believe that because someone fills your mind -LRB- a very weak one , btw -RRB- with stupid `` riddles '' , the kind you learn on elementary school recess , you suddenly come to the `` one truth '' about everything , then you have to kill someone and c

Exception when evaluate data {'x': "I'm the type of guy who loves hood movies from New Jack City to Baby Boy to Killa Season, from the b grade to the Hollywood. but this movie was something different. i am no hater and this movie was kinda enjoyable. but some bits were just weird. well the acting wasn't to good, compared to Silkk The Shockers performance in Hot Boyz (quite good) and Ice-T in new Jack and SVU (great). the scene where Corrupt (Ice-T) kills the wanna be Jamaican dude he says something and lights himself on fire burning both Ice-T and the other dude, this kills the Jamaican, however Ice-T is unharmed, very similar to Ice's other movie Urban Menace (which stars both of these actors) were Snoops character is supernatural, however after this there is nothing suggested that Corrupt is like a demon. When MJ (Silkk) gets stabbed at first he struggling but after that he fights normally and was stabbed in the thigh-WITH OUT BLOOD. and when MJ confesses killing a cop cos the cop wa

Loading parser from serialized file d:\DL project\Post_midsem\data\TProcess.StanfordParser\englishPCFG.ser.gz ... done [0.6 sec].
Parsing file: C:\Users\Priyanka\AppData\Local\Temp\tmpsvr7ucl7
Parsing [sent. 1 len. 439]: I 'm the type of guy who loves hood movies from New Jack City to Baby Boy to Killa Season , from the b grade to the Hollywood . but this movie was something different . i am no hater and this movie was kinda enjoyable . but some bits were just weird . well the acting was n't to good , compared to Silkk The Shockers performance in Hot Boyz -LRB- quite good -RRB- and Ice-T in new Jack and SVU -LRB- great -RRB- . the scene where Corrupt -LRB- Ice-T -RRB- kills the wan na be Jamaican dude he says something and lights himself on fire burning both Ice-T and the other dude , this kills the Jamaican , however Ice-T is unharmed , very similar to Ice 's other movie Urban Menace -LRB- which stars both of these actors -RRB- were Snoops character is supernatural , however after thi

Exception when evaluate data {'x': "Beware, My Lovely (1952) Dir: Harry Horner <br /><br />Production: The Filmmakers/RKO Radio Pictures<br /><br />Credulity-straining thriller from the pioneering producer team of Collier Young and Ida Lupino, aka The Filmmakers (with Lupino pitching in with some uncredited direction). <br /><br />Robert Ryan is the 'peril' and Ida Lupino is the 'woman' in this entry in the 'woman in peril' style film. Ryan plays Howard Wilton, a tightly-wound psychotic handyman drifter (noooo, Ryan? I know, hard to believe). Lupino is the lonely war widow, Helen Gordon, who hires Howard to do some work around her house. Things go downhill from there as Howard makes Helen a prisoner in her own home. <br /><br />Howard has a nasty secret, not that he could reveal it. You see, consciousness is a real challenge for him. Maintaining it, that is. He has an unfortunate habit of coming to and finding his employers dead. This is part of the film's problem. The nature of Howard

Loading parser from serialized file d:\DL project\Post_midsem\data\TProcess.StanfordParser\englishPCFG.ser.gz ... done [0.6 sec].
Parsing file: C:\Users\Priyanka\AppData\Local\Temp\tmpdjoobq_n
Parsing [sent. 1 len. 618]: Beware , My Lovely -LRB- 1952 -RRB- Dir : Harry Horner <br /> <br /> Production : The Filmmakers/RKO Radio Pictures <br /> <br /> Credulity-straining thriller from the pioneering producer team of Collier Young and Ida Lupino , aka The Filmmakers -LRB- with Lupino pitching in with some uncredited direction -RRB- . <br /> <br /> Robert Ryan is the ` peril ' and Ida Lupino is the ` woman ' in this entry in the ` woman in peril ' style film . Ryan plays Howard Wilton , a tightly-wound psychotic handyman drifter -LRB- noooo , Ryan ? I know , hard to believe -RRB- . Lupino is the lonely war widow , Helen Gordon , who hires Howard to do some work around her house . Things go downhill from there as Howard makes Helen a prisoner in her own home . <br /> <br /> Howard has a nast

In [ ]:
# new model

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
t5_model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [11]:
def predict(model, tokenizer, texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
    
    return probs.cpu().numpy(), np.argmax(probs.cpu().numpy(), axis=1)

In [19]:
def generate_context(sentence, num_return=3):
    prompt = f"add a contrasting clause to this sentence without changing its main meaning: {sentence}"
    
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    
    outputs = t5_model.generate(
        **inputs,
        max_length=128,
        num_return_sequences=num_return,
        do_sample=True,
        top_k=50,
        top_p=0.95
    )
    
    results = [
        t5_tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]
    
    return results

In [20]:
def best_attack(sentence, label, model, tokenizer, tries=5):
    candidates = generate_context(sentence, num_return=tries)
    
    best_sentence = sentence
    best_score = 0
    
    for c in candidates:
        probs, _ = predict(model, tokenizer, [c])
        
        score = 1 - probs[0][label]  # reduce true class confidence
        
        if score > best_score:
            best_score = score
            best_sentence = c
    
    return best_sentence

In [21]:
def run_attack(model, tokenizer, samples, max_samples=50):
    success = 0
    total = 0
    
    for sample in tqdm(samples[:max_samples]):
        x = sample["x"]
        y = sample["y"]
        
        probs_orig, pred_orig = predict(model, tokenizer, [x])
        
        # skip incorrect originals
        if pred_orig[0] != y:
            continue
        
        # generate adversarial
        x_adv = best_attack(x, y, model, tokenizer)
        
        probs_adv, pred_adv = predict(model, tokenizer, [x_adv])
        
        if pred_adv[0] != y:
            success += 1
        
        total += 1
    
    return success / total if total > 0 else 0

5 para

In [18]:
models = {}
tokenizers = {}

for name, (model, tokenizer, test_data) in trained_model.items():    
    model.to(device)
    model.eval()
    
    models[name] = model
    tokenizers[name] = tokenizer


for name in models:
    print(f"\nAttacking {name.upper()}...")
    
    asr = run_attack(
        models[name],
        tokenizers[name],
        formatted_data,
        max_samples=50
    )
    
    print(f"Attack Success Rate (ASR): {asr:.4f}")


Attacking BERT...


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [06:12<00:00,  7.44s/it]


Attack Success Rate (ASR): 0.6389

Attacking ELECTRA...


100%|██████████| 50/50 [07:18<00:00,  8.78s/it]


Attack Success Rate (ASR): 0.5122

Attacking ROBERTA...


100%|██████████| 50/50 [08:18<00:00,  9.96s/it]

Attack Success Rate (ASR): 0.7143


In [ ]:
3 para

In [22]:
models = {}
tokenizers = {}

for name, (model, tokenizer, test_data) in trained_model.items():    
    model.to(device)
    model.eval()
    
    models[name] = model
    tokenizers[name] = tokenizer


for name in models:
    print(f"\nAttacking {name.upper()}...")
    
    asr = run_attack(
        models[name],
        tokenizers[name],
        formatted_data,
        max_samples=50
    )
    
    print(f"Attack Success Rate (ASR): {asr:.4f}")


Attacking BERT...


100%|██████████| 50/50 [07:30<00:00,  9.00s/it]


Attack Success Rate (ASR): 0.6944

Attacking ELECTRA...


100%|██████████| 50/50 [08:23<00:00, 10.08s/it]


Attack Success Rate (ASR): 0.5366

Attacking ROBERTA...


100%|██████████| 50/50 [07:18<00:00,  8.78s/it]

Attack Success Rate (ASR): 0.5952


In [ ]:
1 para

In [23]:
models = {}
tokenizers = {}

for name, (model, tokenizer, test_data) in trained_model.items():    
    model.to(device)
    model.eval()
    
    models[name] = model
    tokenizers[name] = tokenizer


for name in models:
    print(f"\nAttacking {name.upper()}...")
    
    asr = run_attack(
        models[name],
        tokenizers[name],
        formatted_data,
        max_samples=50
    )
    
    print(f"Attack Success Rate (ASR): {asr:.4f}")


Attacking BERT...


100%|██████████| 50/50 [08:21<00:00, 10.02s/it]


Attack Success Rate (ASR): 0.6667

Attacking ELECTRA...


100%|██████████| 50/50 [08:38<00:00, 10.37s/it]


Attack Success Rate (ASR): 0.5610

Attacking ROBERTA...


100%|██████████| 50/50 [08:04<00:00,  9.69s/it]

Attack Success Rate (ASR): 0.6905
